# AlphaEarth generation walkthrough (exp023)

Step-by-step interactive run of `configs/exp023_alphaearth_middle_patch.yaml`, calling the same pieces of `gelos.cloud_embeddings` that `gelos.generation` orchestrates in batch.

**Steps:**
1. Build the dataset
2. Compute the footprint manifest
3. Visualize an S2 chip with its AlphaEarth footprint overlaid
4. Resolve the year for each chip
5. Build the AlphaEarth backend
6. Fetch a few example footprints from the S3 mosaic
7. Inspect a fetched 16×16×64 patch
8. Pool to per-chip vectors
9. Plot the embeddings on a map alongside LULC labels

NOTE: This is entirely llm-generated as a sanity check for alphaearth embedding fetching

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rioxarray as rxr
import yaml
from dotenv import load_dotenv
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA

from gelos.cloud_embeddings.aggregate import POOLS
from gelos.cloud_embeddings.backends.alphaearth import AlphaEarthBackend
from gelos.cloud_embeddings.backends.base import FetchRequest
from gelos.cloud_embeddings.footprints import compute_footprint_manifest
from gelos.cloud_embeddings.year import resolve_chip_year
from gelos.generation import instantiate_recursive

## 1. Load config and define paths

`.env` defines `RAW_PATH` and `INTERIM_PATH` the same way docker-compose consumes them, so the same notebook works whether launched via `pixi shell` or in the Docker dev image.

In [ ]:
load_dotenv("../.env")

yaml_path = Path("../configs/exp023_alphaearth_middle_patch.yaml")
raw_data_dir = Path(os.environ["RAW_PATH"])
embedding_dir = Path(os.environ["INTERIM_PATH"])

with open(yaml_path) as f:
    yaml_config = yaml.safe_load(f)

cloud_cfg = yaml_config["cloud_embedding"]
data_version = yaml_config["data_version"]
year_strategy = cloud_cfg["year_strategy"]
patch_size = cloud_cfg["patch_size"]

# Pick the (only) strategy this config defines. The runner loops over
# `extraction_strategies.items()`; this walkthrough does one strategy.
strategy_name, strategy_spec = next(iter(cloud_cfg["extraction_strategies"].items()))
pool_name = strategy_spec.get("pool", "mean")
# Same conditional `out_shape` the runner uses: fixed shape is only needed
# when patches will be flattened (so per-chip vectors stack cleanly).
out_shape = (patch_size, patch_size) if pool_name == "flatten" else None

print(f"experiment: {yaml_config['experiment_name']}")
print(f"data version: {data_version}")
print(f"cloud backend: {cloud_cfg['backend']}")
print(f"strategy: {strategy_name}  pool: {pool_name}  out_shape: {out_shape}")

## 2. Build the dataset

Mirrors `gelos.cloud_embeddings.runner._setup_dataset`: inject `data_root`, instantiate the datamodule recursively from the YAML class_path, then call `setup("predict")` to materialize `datamodule.dataset`.

In [ ]:
yaml_config["data"]["init_args"]["data_root"] = str(raw_data_dir / data_version)
datamodule = instantiate_recursive(yaml_config["data"])
datamodule.setup(stage="predict")
dataset = datamodule.dataset

print(f"chips: {len(dataset)}")
print(f"bands: {dataset.bands}")
print(f"chip-tracker columns: {list(dataset.gdf.columns)}")

In [ ]:
dataset.gdf['aoi_index'].value_counts()

In [ ]:
dataset.gdf = dataset.gdf[dataset.gdf['aoi_index'] == 95]

In [ ]:
len(dataset.gdf)

## 3. Compute the footprint manifest

One row per (chip, strategy, patch). For exp023 the strategy is `middle_patch` with `center_1x1`, so there's exactly one footprint per chip — the central 16×16 pixel block in the reference (S2L2A) raster's CRS.

In [ ]:
manifest = compute_footprint_manifest(
    dataset,
    reference_sensor=cloud_cfg["reference_sensor"],
    patch_size=cloud_cfg["patch_size"],
    strategy_specs=cloud_cfg["extraction_strategies"],
)
print(f"manifest rows: {len(manifest)}  |  CRS: {manifest.crs}")
manifest.head()

## 4. Visualize an S2 chip with its footprint overlaid

Load the raw S2L2A first-timestep GeoTIFF for chip 0 via rioxarray (we get CRS + affine for free), composite RGB from the BLUE/GREEN/RED bands, and overlay the manifest geometry for that chip. The footprint lives in the same UTM CRS as the chip, so no reprojection.

In [ ]:
chip_index = 0
chip_path = dataset._get_file_paths(chip_index, "S2L2A")[0]
s2 = rxr.open_rasterio(chip_path, masked=True)

# Raw S2L2A GeoTIFFs store all 12 bands in S2RTC_BAND_NAMES order:
# COASTAL_AEROSOL(1), BLUE(2), GREEN(3), RED(4), ...
rgb = s2.isel(band=[3, 2, 1]).to_numpy()  # RED, GREEN, BLUE
rgb = np.clip(rgb / 4000.0, 0, 1).transpose(1, 2, 0)
extent = s2.rio.bounds()  # (minx, miny, maxx, maxy)
extent_mpl = (extent[0], extent[2], extent[1], extent[3])

chip_footprints = manifest[
    (manifest.chip_index == chip_index) & (manifest.strategy == strategy_name)
]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb, extent=extent_mpl, origin="upper")
chip_footprints.boundary.plot(ax=ax, color="red", linewidth=2)
ax.set_title(
    f"chip {chip_index} — S2 RGB + AlphaEarth footprint ({len(chip_footprints)} patch(es))"
)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
plt.tight_layout()

## 5. Resolve the year for each chip

AlphaEarth is an *annual* mosaic — each fetch needs a calendar year. The YAML's `year_strategy: from_filename` regex-pulls the year from the S2L2A filename for each chip.

In [ ]:
sample_years = [resolve_chip_year(dataset, i, year_strategy) for i in range(5)]
for i, y in enumerate(sample_years):
    print(f"chip {i}: year={y}  ({dataset._get_file_paths(i, 'S2L2A')[0].name})")

## 6. Build the AlphaEarth backend

Opens (lazily) the TGE Labs GeoZarr mosaic on Source Coop. `cache_dir` is where reprojected per-footprint GeoTIFFs land — once a footprint is fetched, re-running this cell skips S3 entirely for that footprint.

In [ ]:
config_stem = yaml_path.stem
cache_dir = embedding_dir / data_version / config_stem / "_footprint_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

backend = AlphaEarthBackend(cache_dir=cache_dir)
print(f"cache_dir: {cache_dir}")
print(f"mosaic url: {backend.url}")

## 7. Fetch a few example footprints

Mirror `runner.generate_cloud_embeddings`: filter the manifest to the current strategy, then `groupby("chip_index")` and `sort_values(["patch_row", "patch_col"])` inside each group. Build one `FetchRequest` per row in that sorted order, then submit them all as a single `fetch_batch`. Dask's graph optimizer dedupes shared zarr-block reads across requests, so this is faster than serial `fetch` calls. The walkthrough caps to the first `N` chips; the runner's only difference is that it chunks all chips into `window_chips`-sized windows.

In [ ]:
N = 20
strategy_rows = manifest[manifest["strategy"] == strategy_name]

chip_groups = [
    (
        int(chip_index),
        chip_rows.sort_values(["patch_row", "patch_col"]),
        resolve_chip_year(dataset, int(chip_index), year_strategy),
    )
    for chip_index, chip_rows in list(strategy_rows.groupby("chip_index"))[:N]
]

requests = [
    FetchRequest(
        bbox=(
            float(row["bbox_minx"]),
            float(row["bbox_miny"]),
            float(row["bbox_maxx"]),
            float(row["bbox_maxy"]),
        ),
        crs=row["crs"],
        year=year,
        out_shape=out_shape,
    )
    for _, sorted_rows, year in chip_groups
    for _, row in sorted_rows.iterrows()
]

arrays = backend.fetch_batch(requests)
print(f"chips: {len(chip_groups)}  total patch requests: {len(requests)}")
print(f"fetched {len(arrays)} arrays, each {arrays[0].shape} (H, W, C=64 bands)")

## 8. Inspect a fetched AlphaEarth patch

Each fetched patch is `(H, W, 64)` float32 with NaN where the mosaic had nodata. Plot the first three of the 64 latent bands as a pseudo-RGB, plus a per-band mean to see how the embedding varies spatially within the 16×16 footprint.

In [ ]:
patch = arrays[0]
first_chip_index = chip_groups[0][0]
print(f"shape: {patch.shape}  dtype: {patch.dtype}  NaN frac: {np.isnan(patch).mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pseudo_rgb = patch[..., :3]
pseudo_rgb = (pseudo_rgb - np.nanmin(pseudo_rgb)) / (np.nanmax(pseudo_rgb) - np.nanmin(pseudo_rgb) + 1e-9)
axes[0].imshow(pseudo_rgb)
axes[0].set_title("bands 0/1/2 as pseudo-RGB")
axes[0].axis("off")

axes[1].plot(np.nanmean(patch, axis=(0, 1)))
axes[1].set_xlabel("band index")
axes[1].set_ylabel("mean activation")
axes[1].set_title(f"per-band mean (chip {first_chip_index})")
plt.tight_layout()

In [ ]:
chip_index = 0
chip_path = dataset._get_file_paths(chip_index, "S2L2A")[0]
s2 = rxr.open_rasterio(chip_path, masked=True)

# Raw S2L2A GeoTIFFs store all 12 bands in S2RTC_BAND_NAMES order:
# COASTAL_AEROSOL(1), BLUE(2), GREEN(3), RED(4), ...
rgb = s2.isel(band=[3, 2, 1]).to_numpy()  # RED, GREEN, BLUE
rgb = np.clip(rgb / 4000.0, 0, 1).transpose(1, 2, 0)
extent = s2.rio.bounds()  # (minx, miny, maxx, maxy)
extent_mpl = (extent[0], extent[2], extent[1], extent[3])

chip_footprints = manifest[
    (manifest.chip_index == chip_index) & (manifest.strategy == strategy_name)
]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb, extent=extent_mpl, origin="upper")
chip_footprints.boundary.plot(ax=ax, color="red", linewidth=2)
ax.set_title(
    f"chip {chip_index} — S2 RGB + AlphaEarth footprint ({len(chip_footprints)} patch(es))"
)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
plt.tight_layout()

### 8b. Edge-nodata check (post-fix)

Refetch a handful of the section-7 footprints through a backend pointed at a **fresh temp cache dir** (the production `_footprint_cache` holds pre-fix tifs whose 16×16 edges contain false `-128` → NaN). With the padded-selection + bbox-pinned-grid fix in `AlphaEarthBackend`, every patch should come back with **0 NaN cells** (barring genuine mosaic gaps).

In [ ]:
import tempfile

n_check = min(8, len(requests))
edge_check_cache = Path(tempfile.mkdtemp(prefix="ae_edge_check_"))
edge_check_backend = AlphaEarthBackend(cache_dir=edge_check_cache)

edge_check_requests = [
    FetchRequest(bbox=r.bbox, crs=r.crs, year=r.year, out_shape=(16, 16))
    for r in requests[:n_check]
]
edge_check_arrays = edge_check_backend.fetch_batch(edge_check_requests)

print(f"refetched {n_check} section-7 footprints into fresh cache {edge_check_cache}\n")
total_nan = 0
for r, arr in zip(edge_check_requests, edge_check_arrays):
    n_nan = int(np.isnan(arr[..., 0]).sum())
    total_nan += n_nan
    print(f"bbox={tuple(round(v, 1) for v in r.bbox)} {r.crs} year={r.year}: "
          f"{n_nan}/256 NaN cells")
print(f"\ntotal NaN cells across {n_check} patches: {total_nan} (expected 0 post-fix)")

## 9. Overlay the AlphaEarth embedding on the source chip

Drop the fetched patch onto the S2 RGB at its real-world bbox so the embedding lines up spatially with the source data. The patch has 64 channels, so we PCA the per-pixel embedding to 3 components and stretch to 0–1 for a pseudo-RGB rendering — more informative than picking bands 0/1/2 arbitrarily. The red outline is the manifest footprint for the same chip.

In [ ]:
# Pick a chip we actually fetched in step 7.
overlay_chip_index = chip_groups[0][0]
patch_idx_in_arrays = 0  # exp023 has one patch per chip, so chip i → arrays[i]
patch = arrays[patch_idx_in_arrays]

# Source S2 RGB (same recipe as step 4).
chip_path = dataset._get_file_paths(overlay_chip_index, "S2L2A")[0]
s2 = rxr.open_rasterio(chip_path, masked=True)
rgb = s2.isel(band=[3, 2, 1]).to_numpy()
rgb = np.clip(rgb / 4000.0, 0, 1).transpose(1, 2, 0)
s2_bounds = s2.rio.bounds()  # (minx, miny, maxx, maxy)
s2_crs = str(s2.rio.crs)
s2_extent_mpl = (s2_bounds[0], s2_bounds[2], s2_bounds[1], s2_bounds[3])

# Patch bbox in matplotlib extent ordering — same CRS as the S2 chip (UTM).
chip_footprints = manifest[
    (manifest.chip_index == overlay_chip_index) & (manifest.strategy == strategy_name)
]
fp_row = chip_footprints.iloc[patch_idx_in_arrays]
patch_extent_mpl = (
    fp_row["bbox_minx"], fp_row["bbox_maxx"],
    fp_row["bbox_miny"], fp_row["bbox_maxy"],
)

# Fetch the AE embedding for the full chip extent. Same backend/cache as step 7;
# this bbox isn't in _footprint_cache yet (cache key includes bbox), so first
# run reads the zarr; subsequent runs hit the new cache entry.
chip_year = resolve_chip_year(dataset, overlay_chip_index, year_strategy)
full_patch = backend.fetch(bbox=s2_bounds, crs=s2_crs, year=chip_year, out_shape=None)
print(f"full-chip AE patch shape: {full_patch.shape}  NaN frac: {np.isnan(full_patch).mean():.3f}")

# Fit PCA on the full-chip embedding, then apply to both the full chip and the
# 16×16 middle patch so both panels share a color space. Stretch with the
# full-chip min/max so the middle patch sits in the same range visually.
def _apply_pca(arr: np.ndarray, pca: PCA, lo: float, hi: float) -> np.ndarray:
    h, w, c = arr.shape
    flat = arr.reshape(-1, c)
    valid = ~np.isnan(flat).any(axis=1)
    out = np.full((flat.shape[0], 3), np.nan, dtype=np.float32)
    out[valid] = pca.transform(flat[valid])
    return ((out.reshape(h, w, 3) - lo) / (hi - lo + 1e-9))


full_flat = full_patch.reshape(-1, full_patch.shape[-1])
full_valid = ~np.isnan(full_flat).any(axis=1)
pca = PCA(n_components=3).fit(full_flat[full_valid])
full_pca = pca.transform(full_flat[full_valid])
lo, hi = float(full_pca.min()), float(full_pca.max())

full_pca_rgb = _apply_pca(full_patch, pca, lo, hi)
pca_rgb = _apply_pca(patch, pca, lo, hi)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

# Left: S2 RGB + middle-patch outline only (the "before" view).
axes[0].imshow(rgb, extent=s2_extent_mpl, origin="upper")
chip_footprints.boundary.plot(ax=axes[0], color="red", linewidth=2)
axes[0].set_title(f"S2 RGB + patch outline (chip {overlay_chip_index})")

# Middle: S2 RGB with the 16×16 middle-patch AE PCA-RGB dropped in (same PCA + stretch as the full chip).
axes[1].imshow(rgb, extent=s2_extent_mpl, origin="upper")
axes[1].imshow(pca_rgb, extent=patch_extent_mpl, origin="upper", interpolation="nearest")
chip_footprints.boundary.plot(ax=axes[1], color="red", linewidth=2)
axes[1].set_title("S2 RGB + middle-patch AE PCA(1/2/3)")

# Right: full-chip AE PCA-RGB at the chip extent, with the middle-patch outline on top.
axes[2].imshow(full_pca_rgb, extent=s2_extent_mpl, origin="upper", interpolation="nearest")
chip_footprints.boundary.plot(ax=axes[2], color="red", linewidth=2)
axes[2].set_title("Full-chip AE PCA(1/2/3) + patch outline")

for ax in axes:
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
plt.tight_layout()